In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field
from typing_extensions import TypedDict, Annotated

class coarse_classification(BaseModel):
    """Classification result for a single email."""
    coarse_group: str = Field(description="The category assigned to the email among the 5 super groups: transactional, marketing, institutional, personal, needs_human_review")
    confidence: float = Field(description="Model's confidence in the classification, between 0 and 1")


class specialized_agents(BaseModel):
    """Classification result for a single email."""
    category: str = Field(description="The category assigned to the email among the transactional subcategories: finance, travel, orders")
    review_decision: str = Field(description="If agent thinks that the email classification done by the coarse router is wrong, it can suggest the super group that it thinks is correct, so arbiter agent can make the final decision. If the agent agrees with the coarse router, it should return empty string")
    confidence: float = Field(description="Model's confidence in the classification, between 0 and 1")

SEED_CATEGORIES = [
    "work",
    "personal",
    "school",
    "finance",
    "travel",
    "promotions",
    "orders",
    "notifications",
    "newsletters",
]

class EmailState(TypedDict):
    email_id: str
    thread_id: str
    from_name: str
    from_address : str
    subject: str
    body: str
    coarse_group: str | None
    specialist_decision: dict | None
    arbiter_decision: dict | None
    final_category: str | None
    confidence: float | None
    needs_review: bool
    coarse_retrieved_examples: list[dict]
    specialist_retrieved_examples: list[dict]


In [3]:
from dotenv import load_dotenv
load_dotenv("backend/.env")

True

In [4]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.1,
    timeout=60,
    reasoning_format="parsed",
    reasoning_effort="High"
)


In [ ]:
from typing import Any
from copy import deepcopy
from langgraph.graph import StateGraph, START, END
from langgraph.types import RetryPolicy
from langchain_groq import ChatGroq

coarse_llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.1,
    reasoning_format="parsed",
    reasoning_effort="high"
)

specialist_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
)

SOURCE_TAG = "ai-groq-multiagent"
COARSE_GROUPS = {"transactional", "marketing", "institutional", "personal", "needs_human_review"}

SPECIALIST_ALLOWED: dict[str, list[str]] = {
    "transactional": ["finance", "travel", "orders"],
    "marketing": ["promotions", "newsletters"],
    "institutional": ["work", "school", "notifications"],
    "personal": ["personal"],
}

COARSE_BODY_LIMIT = 6000
ARBITER_BODY_LIMIT = 6000
SPECIALIST_BODY_LIMIT = 10000
SPECIALIST_CONFIDENCE_THRESHOLD = 0.70


class arbiter_classification(BaseModel):
    """Final arbiter output for one email."""
    final_category: str = Field(description="Final category among seed categories or uncategorized")
    confidence: float = Field(description="Model confidence between 0 and 1")


def _safe_text(value: Any, limit: int = 6000) -> str:
    if value is None:
        return ""
    return str(value).strip()[:limit]


def _to_float(value: Any, default: float = 0.0) -> float:
    try:
        return float(value)
    except Exception:
        return default


def _normalize_coarse_group(value: Any) -> str:
    raw = str(value or "").strip().lower().replace("-", "_")
    aliases = {
        "transaction": "transactional",
        "institution": "institutional",
        "review": "needs_human_review",
        "human_review": "needs_human_review",
        "needs_review": "needs_human_review",
    }
    normalized = aliases.get(raw, raw)
    if normalized in COARSE_GROUPS:
        return normalized
    return "needs_human_review"


def _normalize_review_decision(value: Any) -> str:
    raw = str(value or "").strip().lower().replace("-", "_")
    if raw in {"", "none", "null", "n/a"}:
        return ""
    return _normalize_coarse_group(raw)


def _normalize_category(value: Any) -> str:
    cat = str(value or "").strip().lower().replace("-", "_")
    if cat in SEED_CATEGORIES:
        return cat
    return "uncategorized"


def _email_body(email_obj: dict[str, Any]) -> str:
    body_text = _safe_text(email_obj.get("body_text"), SPECIALIST_BODY_LIMIT)
    body_html = _safe_text(email_obj.get("body_html"), SPECIALIST_BODY_LIMIT)

    if body_html:
        body = body_html
    elif body_text:
        body = body_text
    else:
        body = ""

    return body


def _build_email_state(state_template: EmailState, email_obj: dict[str, Any], email_no: int) -> EmailState:
    base = deepcopy(state_template)
    email_id = email_obj.get("id") or email_obj.get("gmail_message_id") or str(email_no)

    return {
        "email_id": str(email_id),
        "thread_id": str(email_obj.get("thread_id") or ""),
        "from_name": email_obj.get("from_name"),
        "from_address": email_obj.get("from_address"),
        "subject": email_obj.get("subject"),
        "body": _email_body(email_obj),
        "coarse_group": None,
        "specialist_decision": None,
        "arbiter_decision": None,
        "final_category": None,
        "confidence": None,
        "needs_review": False,
        "coarse_retrieved_examples": list(base.get("coarse_retrieved_examples") or []),
        "specialist_retrieved_examples": list(base.get("specialist_retrieved_examples") or []),
    }


def coarse_router_agent(state: EmailState) -> dict:
    structured_llm = coarse_llm.with_structured_output(coarse_classification)
    coarse_body = _safe_text(state.get("body"), COARSE_BODY_LIMIT)

    prompt = f"""
    You will be provided with emails received by an individual.
    Your task is to classify each email based on its content and sender information, 
    and the confidence level of the classification and determine the appropriate routing for handling the email from 5 super groups:
    1.transactional: emails related to transactions, purchases, orders, receipts, invoices, payments, bookings, reservations, etc.
    2.marketing: emails related to promotions, advertisements, newsletters, offers, discounts, events, weekly updates, etc.
    3.institutional: emails related to organizational or institutional matters, policies, announcements, related to school/uni or work/job indiavidual is at and not promotional etc.
    4.personal: emails related to personal matters, friends, family, etc.
    5.needs_human_review: emails that you are not ceratin about and might require human intervention for proper classification. 
    It is very important to be accurate in your classification, as it will determine how the email is handled.
    Analyze this customer email and classify it:
    
        Email Sender: {state["from_address"]}
        Name: {state["from_name"]}
        Subject: {state["subject"]}
        Body: {coarse_body}
        email_no: {state["email_id"]}
    
    """

    classification = structured_llm.invoke(prompt)
    coarse_group = _normalize_coarse_group(classification.coarse_group)
    confidence = _to_float(classification.confidence)

    needs_review = confidence < 0.70 or coarse_group == "needs_human_review"

    return {
        "coarse_group": coarse_group,
        "confidence": confidence,
        "needs_review": needs_review,
    }


def route_from_coarse(state: EmailState) -> str:
    if state.get("needs_review"):
        return "human_review_agent"

    coarse_group = _normalize_coarse_group(state.get("coarse_group"))
    mapping = {
        "transactional": "transactional_agent",
        "marketing": "marketing_agent",
        "institutional": "institutional_agent",
        "personal": "personal_agent",
    }
    return mapping.get(coarse_group, "human_review_agent")


def _run_specialist_agent(
    state: EmailState,
    *,
    agent_name: str,
    coarse_group: str,
    allowed_categories: list[str],
    category_guidance: str,
) -> dict:
    structured_llm = specialist_llm.with_structured_output(specialized_agents)
    specialist_body = _safe_text(state.get("body"), SPECIALIST_BODY_LIMIT)

    prompt = f"""
    You will be provided with emails received by an individual.
    You are asked to review this mail because a coarse router agent has classified the email into one of 5 super groups: transactional, marketing, institutional, personal, needs_human_review.
    You are the {agent_name} specialist. Your task is to classify each email based on its content and sender information, 
    The coarse router classified this email as: {coarse_group}

    Allowed output categories: {allowed_categories}
    Category guidance:
    {category_guidance}

    You must return:
    - category: one of {allowed_categories}
    - confidence: float [0,1]
    - review_decision: empty string if you agree with the coarse group,
    otherwise one of transactional, marketing, institutional, personal, needs_human_review

    Email Sender: {state["from_address"]}
    Name: {state["from_name"]}
    Subject: {state["subject"]}
    Body: {specialist_body}
    email_no: {state["email_id"]}
"""

    classification = structured_llm.invoke(prompt)

    category = str(classification.category or "").strip().lower().replace("-", "_")
    if category not in allowed_categories:
        category = allowed_categories[0]

    review_decision = _normalize_review_decision(classification.review_decision)
    confidence = _to_float(classification.confidence)

    needs_review = bool(state.get("needs_review"))
    if confidence < SPECIALIST_CONFIDENCE_THRESHOLD:
        needs_review = True
    if review_decision and review_decision != coarse_group:
        needs_review = True

    return {
        "specialist_decision": {
            "agent": agent_name,
            "coarse_group": coarse_group,
            "category": category,
            "review_decision": review_decision,
            "confidence": confidence,
        },
        "confidence": confidence,
        "needs_review": needs_review,
    }


def transactional_agent(state: EmailState) -> dict:
    structured_llm = specialist_llm.with_structured_output(specialized_agents)
    specialist_body = _safe_text(state.get("body"), SPECIALIST_BODY_LIMIT)

    prompt = f"""
    You will be provided with emails received by an individual. 
    You are asked to review this mail because a coarse router agent has classified the email into one of 5 super groups: transactional, marketing, institutional, personal, needs_human_review. You are a specialized agent for handling emails classified as transactional.
    Your task is to classify each email based on its content and sender information, 
    and the confidence level of the classification and determine the appropriate category:
    1.finance: emails related to financial transactions, banking, investments, credit cards, loans, etc.
    2.travel: emails related to travel bookings, reservations, itineraries, flights, hotels, etc.
    3.orders: emails related to purchases, orders, delivery status,etc.
    It is very important to be accurate in your classification, as it will determine how the email is handled.
    Analyze this customer email and classify it:
    
        Email Sender: {state["from_address"]}
        Name: {state["from_name"]}
        Subject: {state["subject"]}
        Body: {specialist_body}
        email_no: {state["email_id"]}
        
    Give output in following format:
    category: one of finance, travel, orders
    confidence: a float between 0 and 1 indicating confidence in the classification
    review_decision: if you think the coarse router agent has made a wrong classification and this email does not belong to transactional super group, please suggest the super group that you think is correct among the 5 super groups: transactional, marketing, institutional, personal, needs_human_review. If you agree with the coarse router's classification and think this email belongs to transactional super group, please return "" empty string as review_decision.
    """

    classification = structured_llm.invoke(prompt)

    category = str(classification.category or "").strip().lower().replace("-", "_")
    if category not in SPECIALIST_ALLOWED["transactional"]:
        category = "orders"

    review_decision = _normalize_review_decision(classification.review_decision)
    confidence = _to_float(classification.confidence)

    needs_review = bool(state.get("needs_review"))
    if confidence < SPECIALIST_CONFIDENCE_THRESHOLD:
        needs_review = True
    if review_decision and review_decision != "transactional":
        needs_review = True

    return {
        "specialist_decision": {
            "agent": "transactional_agent",
            "coarse_group": "transactional",
            "category": category,
            "review_decision": review_decision,
            "confidence": confidence,
        },
        "confidence": confidence,
        "needs_review": needs_review,
    }


def marketing_agent(state: EmailState) -> dict:
    return _run_specialist_agent(
        state,
        agent_name="marketing_agent",
        coarse_group="marketing",
        allowed_categories=SPECIALIST_ALLOWED["marketing"],
        category_guidance=(
            "promotions: sales, discounts, product announcements, campaign emails; "
            "newsletters: recurring digest/roundup/subscriber editorial updates"
        ),
    )


def institutional_agent(state: EmailState) -> dict:
    return _run_specialist_agent(
        state,
        agent_name="institutional_agent",
        coarse_group="institutional",
        allowed_categories=SPECIALIST_ALLOWED["institutional"],
        category_guidance=(
            "work: job/project/team communication; "
            "school: classes, assignments, grades, university administration; "
            "notifications: system/app alerts, OTP/security/event notifications"
        ),
    )


def personal_agent(state: EmailState) -> dict:
    return _run_specialist_agent(
        state,
        agent_name="personal_agent",
        coarse_group="personal",
        allowed_categories=SPECIALIST_ALLOWED["personal"],
        category_guidance="personal: direct personal communication from real people",
    )


def human_review_agent(state: EmailState) -> dict:
    return {
        "needs_review": True,
        "specialist_decision": {
            "agent": "human_review_agent",
            "coarse_group": _normalize_coarse_group(state.get("coarse_group")),
            "category": "",
            "review_decision": "needs_human_review",
            "confidence": 0.0,
        },
    }


def route_after_specialist(state: EmailState) -> str:
    if bool(state.get("needs_review")):
        return "arbiter_agent"

    specialist = state.get("specialist_decision") or {}
    coarse_group = _normalize_coarse_group(state.get("coarse_group"))
    review_decision = _normalize_review_decision(specialist.get("review_decision"))
    specialist_confidence = _to_float(specialist.get("confidence"))
    specialist_category = _normalize_category(specialist.get("category"))

    if review_decision and review_decision != coarse_group:
        return "arbiter_agent"
    if specialist_confidence < SPECIALIST_CONFIDENCE_THRESHOLD:
        return "arbiter_agent"
    if specialist_category not in SEED_CATEGORIES:
        return "arbiter_agent"

    return "finalize_without_arbiter"


def finalize_without_arbiter(state: EmailState) -> dict:
    specialist = state.get("specialist_decision") or {}
    category = _normalize_category(specialist.get("category"))
    confidence = _to_float(specialist.get("confidence"))

    return {
        "final_category": category,
        "arbiter_decision": {
            "final_category": category,
            "confidence": confidence,
            "resolved_by": "specialist",
        },
    }


def arbiter_agent(state: EmailState) -> dict:
    structured_llm = coarse_llm.with_structured_output(arbiter_classification)
    specialist = state.get("specialist_decision") or {}
    arbiter_body = _safe_text(state.get("body"), ARBITER_BODY_LIMIT)

    prompt = f"""
    You will be provided with emails received by an individual. 
    You are asked to review this mail because a coarse router agent has classified the email into one of 5 super groups: transactional, marketing, institutional, personal, needs_human_review. 
    Then a specialized cateogry agent categroized it into a particular category.
    You are the final arbiter for personal inbox classification. IT might be the case that coarse router agent has made a wrong classification and assigned the email to wrong super group, and as a result the email was handled by wrong specialist agent and categorized into wrong category. Your task is to review the email, the coarse router's classification, and the specialist agent's classification, and make the final decision on what category this email belongs to among seed categories or uncategorized.
    But also take the decision by previous agents into account  and try to overrule them without good reason.
    Decide exactly one final category from:
    {SEED_CATEGORIES + ["uncategorized"]}

    It is very important to be accurate in your classification, as it will determine how the email is handled.
    Analyze this customer email and classify it:
        Email Sender: {state["from_address"]}
        Name: {state["from_name"]}
        Subject: {state["subject"]}
        Body: {arbiter_body}
        email_no: {state["email_id"]}
        Coarse router decision: {state.get("coarse_group")}
        Coarse confidence: {state.get("confidence")}
        Specialist decision: {specialist}
        Needs review: {state.get("needs_review")}
    Give output in following format:
    final_category: one of {SEED_CATEGORIES + ["uncategorized"]}
    confidence: a float between 0 and 1 indicating confidence in the classification
"""

    classification = structured_llm.invoke(prompt)
    final_category = _normalize_category(classification.final_category)

    specialist_category = _normalize_category((specialist or {}).get("category"))
    if final_category == "uncategorized" and specialist_category in SEED_CATEGORIES:
        final_category = specialist_category

    confidence = _to_float(classification.confidence)

    return {
        "final_category": final_category,
        "arbiter_decision": {
            "final_category": final_category,
            "confidence": confidence,
            "resolved_by": "arbiter",
        },
        "confidence": confidence,
        "needs_review": True,
    }


def classify_emails_batch(state: EmailState, email_objs) -> list[tuple[str, str, bool]]:
    """Run full multi-agent graph per email and return [(category, source, needs_review), ...]."""
    if not email_objs:
        return []

    if "app" not in globals():
        raise RuntimeError("Run the workflow compile cell first so `app` is available.")

    results: list[tuple[str, str, bool]] = []

    for i, e in enumerate(email_objs, 1):
        email_state = _build_email_state(state, e, i)
        final_state = app.invoke(email_state)

        category = _normalize_category(final_state.get("final_category"))
        needs_review = bool(final_state.get("needs_review", False))

        results.append((category, SOURCE_TAG, needs_review))

    return results



GroqError: The api_key client option must be set either by passing api_key to the client or by setting the GROQ_API_KEY environment variable

In [ ]:
# Specialists and arbiter are fully implemented in the previous cell.
# Quick sanity view:
SPECIALIST_ALLOWED



In [ ]:
# Build and compile the complete multi-agent graph
workflow = StateGraph(EmailState)

workflow.add_node(
    "coarse_router_agent",
    coarse_router_agent,
    retry_policy=RetryPolicy(max_attempts=3)
)
workflow.add_node("transactional_agent", transactional_agent)
workflow.add_node("marketing_agent", marketing_agent)
workflow.add_node("institutional_agent", institutional_agent)
workflow.add_node("personal_agent", personal_agent)
workflow.add_node("human_review_agent", human_review_agent)
workflow.add_node("finalize_without_arbiter", finalize_without_arbiter)
workflow.add_node("arbiter_agent", arbiter_agent)

workflow.add_edge(START, "coarse_router_agent")
workflow.add_conditional_edges(
    "coarse_router_agent",
    route_from_coarse,
    {
        "transactional_agent": "transactional_agent",
        "marketing_agent": "marketing_agent",
        "institutional_agent": "institutional_agent",
        "personal_agent": "personal_agent",
        "human_review_agent": "human_review_agent",
    },
)

workflow.add_conditional_edges(
    "transactional_agent",
    route_after_specialist,
    {
        "arbiter_agent": "arbiter_agent",
        "finalize_without_arbiter": "finalize_without_arbiter",
    },
)
workflow.add_conditional_edges(
    "marketing_agent",
    route_after_specialist,
    {
        "arbiter_agent": "arbiter_agent",
        "finalize_without_arbiter": "finalize_without_arbiter",
    },
)
workflow.add_conditional_edges(
    "institutional_agent",
    route_after_specialist,
    {
        "arbiter_agent": "arbiter_agent",
        "finalize_without_arbiter": "finalize_without_arbiter",
    },
)
workflow.add_conditional_edges(
    "personal_agent",
    route_after_specialist,
    {
        "arbiter_agent": "arbiter_agent",
        "finalize_without_arbiter": "finalize_without_arbiter",
    },
)

workflow.add_edge("human_review_agent", "arbiter_agent")
workflow.add_edge("finalize_without_arbiter", END)
workflow.add_edge("arbiter_agent", END)

# compile without checkpointer for simple notebook execution
app = workflow.compile()



In [9]:
email_objs= [
  {
    "id": "5038a6b9-4eee-4d8a-ab99-6b1489aca975",
    "gmail_message_id": "19d09d959075262e",
    "from_name": "Adobe Acrobat",
    "from_address": "mail@mail.adobe.com",
    "subject": "Keep your PDFs safe from prying eyes 👀",
    "body_text": "View web version:\r\nhttps://t3.mail.adobe.com/r/?id=t3f04bd6b-5233-48c0-8bfb-a5851e7a786c,14842bfc,2ac92f&e=cDE9JTQwQlNlZ",
    "body_html": "\r\n\r\n\r\n<!DOCTYPE html>\r\n<html xmlns=\"https://www.w3.org/1999/xhtml\" xmlns:v=\"urn:schemas-microsoft-com:vml\" xmlns:o=\"urn:"
  },
  {
    "id": "4f077715-32ad-488c-b151-833bca4e9db9",
    "gmail_message_id": "19d0971033732473",
    "from_name": "Weee!",
    "from_address": "hello@mktg.sayweee.com",
    "subject": "🍓🥒Fresh Asian Produce Delivered Right to Your Doorstep!",
    "body_text": "[speaker]\r\n\r\nBack in stock\r\n\r\n[carat](https://click.sayweee.com/grocery-near-me/delivery/back-in-stock?mktOrg=weee_26&mk",
    "body_html": "<!DOCTYPE html><html xmlns=\"http://www.w3.org/1999/xhtml\" xmlns:o=\"urn:schemas-microsoft-com:office:office\" xmlns:v=\"urn"
  },
  {
    "id": "c80bbcaf-5657-40d6-833a-95a85ac94f95",
    "gmail_message_id": "19d0935a29514a42",
    "from_name": "Gap Factory",
    "from_address": "gapfactory@email.gapfactory.com",
    "subject": "New deal: 50% off polos, dresses, and pants",
    "body_text": " \r\n\r\n \r\n\r\n New deal: 50% off polos, dresses, and pants \r\n\r\n \r\n\r\n\r\n \r\n\r\n \r\n\r\n \r\n\r\n \r\n \r\n  \r\n  \r\n \r\n\r\nhttps://click.email.",
    "body_html": " \r\n\r\n \r\n\r\n<!DOCTYPE html PUBLIC \"-//W3C//DTD XHTML 1.0 Transitional//EN\" \"http://www.w3.org/TR/xhtml1/DTD/xhtml1-transit"
  },
  {
    "id": "b0658652-73cf-4661-8f65-85aaa30e93b3",
    "gmail_message_id": "19d09342489fc15e",
    "from_name": "adidas",
    "from_address": "adidas@us-news.comms.adidas.com",
    "subject": "Inspired by florals and soccer - adidas x Liberty London",
    "body_text": "",
    "body_html": "<!DOCTYPE html PUBLIC \"-//W3C//DTD HTML 4.01 Transitional//EN\" \"http://www.w3.org/TR/html4/loose.dtd\">\r\n<html lang=\"en\">"
  },
  {
    "id": "2d7a91f7-c65a-4c49-845d-af459515426f",
    "gmail_message_id": "19d08c588849e5cc",
    "from_name": "Screener.in",
    "from_address": "no-reply@screener.in",
    "subject": "Screener.in - Watchlist updates",
    "body_text": "\r\n\r\nLatest Updates:\r\n\r\n  \r\n  \r\n  ION Exchange:\r\n    Announcement under Regulation 30 (LODR)-Credit Rating\r\n  \r\n  HCL Tec",
    "body_html": "<div style=\"line-height: 1.30; font-family: sans-serif;\">\r\n  \r\n\r\n\r\n  \r\n    <h1>Latest updates</h1>\r\n\r\n    <ul style=\"lis"
  },
  {
    "id": "3a86bed1-c01d-4dd1-be96-c330545d1ecc",
    "gmail_message_id": "19d088d623420dd0",
    "from_name": "SHEIN",
    "from_address": "shein@news.edmmarket.shein.com",
    "subject": "Fresh Drip & Fire Deals 😎",
    "body_text": "",
    "body_html": "   \n<!DOCTYPE html>\n<html xmlns=\"https://www.w3.org/1999/xhtml\" xmlns:v=\"urn:schemas-microsoft-com:vml\" xmlns:o=\"urn:sch"
  },
  {
    "id": "45f26583-414b-4dce-bfcd-9ab333dffca4",
    "gmail_message_id": "19d0887a4d85df47",
    "from_name": "SHEIN",
    "from_address": "shein@market-us.shein.com",
    "subject": "Only three days left to use your 30%OFF coupon!",
    "body_text": "",
    "body_html": "<!doctype html>\r\n<html xmlns=\"http://www.w3.org/1999/xhtml\" xmlns:v=\"urn:schemas-microsoft-com:vml\" xmlns:o=\"urn:schemas"
  },
  {
    "id": "e67852b7-7015-4cce-9625-950166466b56",
    "gmail_message_id": "19d086d6e3d32fa7",
    "from_name": "Foot Locker",
    "from_address": "footlocker@bc.footlocker.com",
    "subject": "App-Only Sale coming soon!",
    "body_text": "",
    "body_html": "<!DOCTYPE html>\r\n<html>\r\n<head>\r\n<meta name=\"viewport\" content=\"width=device-width\">\r\n<meta http-equiv=\"Content-Type\" co"
  },
  {
    "id": "32f0e57b-4255-49ff-8fa3-bc80188768c5",
    "gmail_message_id": "19d0862dc1dc700c",
    "from_name": "Ryan ＠ Fabletics",
    "from_address": "fabletics@emails.fabletics.com",
    "subject": "💬 Hey there!",
    "body_text": "You may have missed this  ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ \n͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏‌ ͏",
    "body_html": "<!DOCTYPE html><html xmlns:v=\"urn:schemas-microsoft-com:vml\" xmlns:o=\"urn:schemas-microsoft-com:office:office\" lang=\"en\""
  },
  {
    "id": "4f2ad9eb-5591-43ec-99b3-2e1807c457cb",
    "gmail_message_id": "19d08074f1d619a1",
    "from_name": "Stevens Institute of Technology",
    "from_address": "graduate@stevens.edu",
    "subject": "The Stevens Graduate Virtual Open House is in 2 weeks!",
    "body_text": " Hi Parva Vasudevbhai,\r\n\r\n Graduate Admissions at Stevens Institute of Technology would like to invite you to join us on",
    "body_html": "<!DOCTYPE html>\r\n<html>\r\n<head>\r\n<meta content=\"text/html; charset=utf-8\" http-equiv=\"content-type\" />\r\n<base target=\"_b"
  }
]
state: EmailState = {
    "email_id": "",
    "thread_id": "",
    "from_name": "",
    "from_address": "",
    "subject": "",
    "body": "",
    "coarse_group": None,
    "specialist_decision": None,
    "arbiter_decision": None,
    "final_category": None,
    "confidence": None,
    "needs_review": False,
    "retrieved_examples": [],
}
a = classify_emails_batch(state, email_objs)

marketing_agent
marketing_agent
marketing_agent
marketing_agent
marketing_agent
marketing_agent
marketing_agent
marketing_agent
marketing_agent
marketing_agent


In [7]:
state: EmailState = {
    "email_id": "",
    "thread_id": "",
    "from_name": "",
    "from_address": "",
    "subject": "",
    "body": "",
    "coarse_group": None,
    "specialist_decision": None,
    "arbiter_decision": None,
    "final_category": None,
    "confidence": None,
    "needs_review": False,
    "retrieved_examples": [],
}

In [11]:
email_objs= [
# {
#     "id": "5038a6b9-4eee-4d8a-ab99-6b1489aca975",
#     "gmail_message_id": "19d09d959075262e",
#     "from_name": "CSE 598: Special Topics (2026 Spring C)",
#     "from_address": "notifications@instructure.com",
#     "subject": "Submission Posted: Quiz 7, CSE 598: Special Topics (2026 Spring C)",
#     "body_html": """<!DOCTYPE html>
# <html dir="ltr" lang="en">
# <head>
#   <meta name="viewport" content="width=device-width">
#   <meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
#   <style type="text/css">
# /*
# Changes to font size (14->16) for smaller screens
# table[class=body] is the only selector that works for all vendors
# */
# @media only screen and (max-width: 620px) {
#   table[class=body] p,
#   table[class=body] ul,
#   table[class=body] ol,
#   table[class=body] td,
#   table[class=body] span,
#   table[class=body] a {
#     font-size: 16px !important;
#   }
#   /* remove padding for mobile so no gray shows */
#   table[class=body] .bodycell {
#     padding: 0 !important;
#     width: 100% !important;
#   }
#   /* reduce padding from 20->10 for mobile */
#   table[class=body] .maincell {
#     padding: 10px !important;
#   }
# }
# /*
# ExternalClass fixes Outlook.com / Hotmail emails
# */
# @media all {
#   .ExternalClass {
#     width: 100%;
#   }
#   .ExternalClass,
#   .ExternalClass p,
#   .ExternalClass span,
#   .ExternalClass font,
#   .ExternalClass td,
#   .ExternalClass div {
#     line-height: 100%;
#   }
# }
#   </style>
# </head>
# <!--
# background: white (could be gray)
# default sans serif fonts, 14px, 1.3, #444444
# vendor prefixes for Outlook (-ms) and iOS (-webkit)
# Margin is capitalized to fix Outlook.com
# -->
# <body class="" style="background-color:#ffffff; font-family:'Open Sans', 'Lucida Grande', 'Segoe UI', Arial, Verdana, 'Lucida Sans Unicode', Tahoma, 'Sans Serif'; font-size:14px; color: #444444; line-height:1.3; Margin:0; padding:0; -ms-text-size-adjust:100%; -webkit-font-smoothing:antialiased; -webkit-text-size-adjust:100%;">

#   <!-- body: background table (if body has a color, this should match) -->
#   <table border="0" cellpadding="0" cellspacing="0" class="body" style="border-collapse:separate; background-color:#ffffff; width:100%; box-sizing:border-box; mso-table-lspace:0pt; mso-table-rspace:0pt;">
#     <tr>
#       <!-- width and max-width so it can scale for mobile -->
#       <td class="bodycell" style="max-width:600px; width:100%; font-family:'Open Sans', 'Lucida Grande', 'Segoe UI', Arial, Verdana, 'Lucida Sans Unicode', Tahoma, 'Sans Serif'; font-size:14px; vertical-align:top; display:block; box-sizing:border-box; padding:10px; Margin:0 auto !important;">

# <!-- for older versions of Outlook that don't support max-width -->
# <!--[if (gte mso 9)|(IE)]>
# <table width="600" align="center" cellpadding="0" cellspacing="0" border="0"><tr><td>
# <![endif]-->

#         <!-- main: white box for content -->
#         <table class="main" style="background:#fff; width:100%; border-collapse:separate; mso-table-lspace:0pt; mso-table-rspace:0pt; ">
#           <tr>
#             <td class="maincell" style="font-family:sans-serif; font-size:14px; vertical-align:top; box-sizing:border-box; padding:20px;">

                    
# <p>Your instructor has released grade changes and new comments for Quiz 7. These changes are now viewable.</p>

#   <p>graded: Mar 26 at 10:08pm</p>


# <p></p>


#             </td>
#           </tr>
#         </table>
#         <!-- /.main -->

#         <!-- logo: branding -->
#         <table class="logo" style="width:100%; box-sizing:border-box; border-collapse:separate; mso-table-lspace:0pt; mso-table-rspace:0pt; ">
#           <tr>
#             <td class="logocell" style="text-align:center; vertical-align:top; box-sizing:border-box; padding:10px;">
#               <img src="https://du11hjcvx0uqb.cloudfront.net/dist/images/email_signature-d2c5880612.png" alt="">
#             </td>
#           </tr>
#         </table>
#         <!-- /.logo -->

#         <!-- footer: gray text below main -->
#         <table class="footer" style="width:100%; box-sizing:border-box; border-collapse:separate; mso-table-lspace:0pt; mso-table-rspace:0pt; ">
#           <tr>
#             <td class="footercell" style="font-family:sans-serif; font-size:14px; vertical-align:top; color:#a8b9c6; font-size:12px; text-align:center; padding:10px; box-sizing:border-box; ">

#                 <a href="https://canvas.asu.edu/courses/250261/assignments/7301451/submissions/1321209">
#     You can view it here
#   </a> &nbsp;|&nbsp;

#                 <a href="https://canvas.asu.edu/profile/communication" style="white-space: nowrap;">Update your notification settings</a>

#             </td>
#           </tr>
#         </table>
#         <!-- /.footer -->

# <!--[if (gte mso 9)|(IE)]>
# </td></tr></table>
# <![endif]-->

#       </td>
#     </tr>
#   </table>
#   <!-- /.body -->

# </body>
# </html>"""
# },
{
    "id": "5038a6b9-4eee-4d8a-ab99-6b1489aca975232323",
    "gmail_message_id": "19d09d959075262e23",
    "from_name": "Stevens Institute of Technology",
    "from_address": "graduate@stevens.edu",
    "subject": "Reminder: Register for Our March 31 Current Student Panel",
    "body_html": """<!DOCTYPE html>
<html>
<head>
<meta content=3D"text/html; charset=3Dutf-8" http-equiv=3D"content-type" />
<base target=3D"_blank" />
<style>body { font-family: Times New Roman, Times, serif; font-size: 12pt; =
} a, a:link, a:visited { color: blue; } a:active, a:hover { color: red; } p=
re { white-space: pre-wrap; }</style><meta name=3D"viewport" content=3D"wid=
th=3Ddevice-width, initial-scale=3D1.0" /><meta http-equiv=3D"Content-Type"=
 content=3D"text/html; charset=3Dus-ascii" /><style type=3D"text/css">/* Ba=
se body background */
body {
  margin: 0;
  padding: 0;
  width: 100% !important;
  -webkit-text-size-adjust: 100%;
  -ms-text-size-adjust: 100%;
  background-color: #FAFAFA;
}

/* Card-style container for main content + footer */
.card-container {
  border-radius: 12px;
  box-shadow: 0 4px 16px rgba(0,0,0,0.12);
  overflow: hidden;
}

/* Inner card table */
.card-inner {
  border-radius: 12px;
}

/* Modern CTA button baseline */
.cta-button {
  display: inline-block;
  font-family: Arial, Helvetica, sans-serif;
  font-size: 16px;
  font-weight: bold;
  text-align: center;
  text-decoration: none !important;
  border-radius: 999px;
  padding: 12px 32px;
}

/* Helper class for centering nested tables in footer on desktop */
.footer-center {
  margin-left: auto !important;
  margin-right: auto !important;
}

/* Force Gmail mobile to center the footer content */
.footer-fullwidth {
  margin-left: 0 !important;
  margin-right: 0 !important;
  padding: 0 !important;
}

.footer-fullwidth > table {
  margin: 0 auto !important;
}

/* Mobile styles */
@media only screen and (max-width: 600px) {
  img {
    max-width: 100%;
    height: auto;
  }

  p {
    vertical-align: top !important;
    font-family: Arial, Helvetica, sans-serif !important;
    padding-right: 10px !important;
    font-size: 15px !important;
    line-height: 1.6 !important;
  }

  .department {
    font-size: 18px !important;
  }

  .wrapper {
    width: 100% !important;
  }

  .title {
    font-size: 20px !important;
  }

  .logo {
    width: auto !important;
    display: block !important;
    margin-right: 0 !important;
  }

  .info {
    width: auto !important;
    display: block !important;
  }

  .cta {
    margin: 8px;
    cursor: pointer;
    display: inline-block !important;
    font-family: Arial, Helvetica, sans-serif !important;
    font-weight: bold !important;
    font-size: 16px !important;
    text-align: center;
    text-decoration: none !important;
  }

  .infotable {
    width: 50% !important;
  }

  .speaker-column {
    display: block !important;
    width: 100% !important;
    padding-left: 0 !important;
    padding-right: 0 !important;
    padding-top: 16px !important;
  }

  .speaker-column-first {
    padding-top: 16px !important;
  }

  .speaker-headshot {
    width: 60px !important;
    height: auto !important;
  }

  .speaker-name {
    margin: 0 !important;
    line-height: 1.2 !important;
  }

  .speaker-title {
    margin: 1px 0 0 0 !important;
    line-height: 1.2 !important;
  }
}

</style>
</head>
<body bgcolor=3D"#FAFAFA" leftmargin=3D"0" marginheight=3D"0" marginwidth=
=3D"0" topmargin=3D"0">
<div style=3D"display: none; font-size: 1px; color: #333333; line-height: 1=
px; max-height: 0px; max-width: 0px; opacity: 0; overflow: hidden;">Tuesday=
, March 31, 2026 </div><!--=3D START FULL PAGE TABLE=3D=3D=3D=3D=3D=3D=3D=
=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D--><table bgcolor=3D"#FAFAFA" =
cellpadding=3D"0" cellspacing=3D"0" style=3D"margin: 0; padding: 0;" width=
=3D"100%"><tbody><tr><td align=3D"center" style=3D"padding: 20px 0;"><!-- C=
ard container including main content + footer --><table align=3D"center" ce=
llpadding=3D"0" cellspacing=3D"0" class=3D"card-container" style=3D"margin:=
 0 auto;" width=3D"600"><tbody><tr><td><!--=3D START WRAPPER TABLE=3D=3D=3D=
=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D--><table align=3D=
"center" bgcolor=3D"#FFFFFF" cellpadding=3D"0" cellspacing=3D"0" class=3D"c=
ard-inner" style=3D"margin: 0 auto; padding-right: 0px; padding-left: 0px;"=
 width=3D"600"><tbody><tr><td><!--=3D STEVENS HEADER=3D=3D=3D=3D=3D=3D=3D=
=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D--><table align=3D"center" cel=
lpadding=3D"0" cellspacing=3D"0" style=3D"line-height:0; margin:0; padding:=
0; border-collapse:collapse;" width=3D"100%"><tbody><tr><td style=3D"font-s=
ize:0; line-height:0; padding:0; margin:0;"><img alt=3D"Stevens Institute o=
f Technology Graduate Admissions" style=3D"width:100%; display:block; borde=
r:0; margin:0; padding:0;" width=3D"100%" src=3D"https://mx.technolutions.n=
et/proxy/V7rENby-I3dRIKMcLT7ieOthm0veNcvNSQGN3qDqofPCc3O-II0TThs4ceHsEkaP3G=
USCUtYR9AzV5khO2BXL950YvqS34Z1PbgAIZe8ffnbO9O_JkZWJop4fkfMHcIr/Otb4NcW_L_4t=
JHdyNW1yXg" /></td></tr><tr><td style=3D"font-size:0; line-height:0; paddin=
g:0; margin:0;"><img alt=3D"Graduate Admissions header image" style=3D"widt=
h: 100%; display: block; border: 0px; margin: 0px; padding: 0px;" width=3D"=
100%" src=3D"https://mx.technolutions.net/proxy/V7rENby-I3dRIKMcLT7ieOthm0v=
eNcvNSQGN3qDqofPCc3O-II0TThs4ceHsEkaP3GUSCUtYR9AzV5khO2BXL_CppIbzsvgS5ewfFL=
qPtTRPdp5l0Lo2BcXybw6C3f-Y/FqU242AinSm7Ek-F7fLR7z9LtvZhI9kkbs5cqPK3_3WUkA9-=
vOCcKiNgf1R9YtJZ" /></td></tr></tbody></table><!--=3D SCHOOL HEADER=3D=3D=
=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D--><table align=
=3D"center" cellpadding=3D"0" cellspacing=3D"0" width=3D"100%"><tbody><tr><=
td style=3D"margin: 0px; padding-top: 10px; padding-bottom: 10px; text-alig=
n: left; vertical-align: top; color: rgb(73, 72, 73); line-height: 1.4;"><t=
able align=3D"center" border=3D"0" cellpadding=3D"25" cellspacing=3D"1" wid=
th=3D"100%"><tbody><tr><td><p style=3D"font-weight: normal;"><span style=3D=
"font-family:Arial,Helvetica,sans-serif; font-size:14px;">Hi Parva Vasudevb=
hai,</span></p><p style=3D"font-weight: normal;"><span style=3D"font-family=
:Arial,Helvetica,sans-serif; font-size:14px;">Join us on <strong>Tuesday, M=
arch 31</strong> for our upcoming <strong>Current Student Panel webinar</st=
rong>, featuring graduate students from the Stevens School of Business and =
the Schaefer School of Engineering.</span></p><p style=3D"font-weight: norm=
al;"><span style=3D"font-family:Arial,Helvetica,sans-serif; font-size:14px;=
">Whether you are still exploring Stevens or preparing for your next step a=
s an applicant, this interactive session is a great opportunity to hear dir=
ectly from current students about academics, student life, and how Stevens =
is preparing them for their careers.</span></p><p style=3D"font-weight: bol=
d; text-align: center; margin: 0 0 18px 0;"><span style=3D"font-family:Aria=
l,Helvetica,sans-serif; font-size:16px; color:#A32638;">Reserve your spot f=
or March 31.</span></p><table align=3D"center" border=3D"0" cellpadding=3D"=
0" cellspacing=3D"0" style=3D"margin: 0 auto 30px auto;"><tbody><tr><td ali=
gn=3D"center" bgcolor=3D"#A32638" style=3D"border-radius: 999px;"><a href=
=3D"https://mx.technolutions.net/ss/c/u001.Jg_sqgPJs1lCjHcCRsBB8fEl66DSe5Mn=
VK3g7OCgm1ZUR_wmc8ZzBZGFjYUCHtz0tk8K5E7UePlVwxOnl4Ggp8chC8BK9NmjruMnVWACynL=
npyv4wJRpjSmQBN3NloN7gUhMdc9BJZFydkDW61BXD6gUr0NLMqLfcTLflrnjRUo1mEMwUFkmEy=
NeA0DP522r/4pa/VNKrvYH6Rm6eWVDnmUWojA/h0/h001.6bIoFpyPHY7lW5pJ1_yk4G4K83nkL=
PiLV0wuoXelLEQ" title=3D"https://gradadmissions.stevens.edu/apply/form?id=
=3D454f478e-df04-411e-84c0-3f71ddf39075&amp;PERSON=3Da020c992-fa96-416b-af1=
4-8be4ea2aec4f" class=3D"cta-button" style=3D"color: #FFFFFF; display: inli=
ne-block; font-family: Arial, Helvetica, sans-serif; font-size: 16px; font-=
weight: bold; text-align: center; text-decoration: none !important; padding=
: 12px 32px; border-radius: 999px;">Register Now</a></td></tr></tbody></tab=
le><p style=3D"font-weight: bold; font-family: Arial, Helvetica, sans-serif=
; font-size: 14px; margin-top: 20px;"><span style=3D"font-size:20px;">Featu=
red Speakers</span></p><p style=3D"font-weight: normal; margin-top: 0;"><sp=
an style=3D"font-family:Arial,Helvetica,sans-serif; font-size:14px;">Hear f=
rom members of the Stevens community who can share firsthand insight into t=
he graduate student experience.</span></p><table border=3D"0" cellpadding=
=3D"0" cellspacing=3D"0" style=3D"width:100%; font-family: Arial, Helvetica=
, sans-serif; font-size: 13px;"><tbody><tr><td class=3D"speaker-column spea=
ker-column-first" style=3D"width:50%; padding-right:10px; vertical-align:to=
p;"><table border=3D"0" cellpadding=3D"0" cellspacing=3D"0" style=3D"width:=
100%;"><tbody><tr><td style=3D"width:60px; vertical-align:top;"><img alt=3D=
"Headshot of Diksha Kishore" class=3D"speaker-headshot" style=3D"width: 60p=
x; height: 60px; border-radius: 50%; display: block;" height=3D"60" width=
=3D"60" src=3D"https://mx.technolutions.net/proxy/V7rENby-I3dRIKMcLT7ieOthm=
0veNcvNSQGN3qDqofPCc3O-II0TThs4ceHsEkaP3GUSCUtYR9AzV5khO2BXL-jvLICqfsYBAKnk=
pUUvTDg8L05NzGGWhFTXPXZUnU6P/ozXN5QT8h5zWAqTer3wKvg" /></td><td style=3D"pa=
dding-left:8px; vertical-align:top;"><p class=3D"speaker-name" style=3D"mar=
gin:0; font-weight:bold; font-size:13px;">
                                                Graduate Admissions
                                              </p></td></tr></tbody></table=
></td><td class=3D"speaker-column speaker-column-first" style=3D"width:50%;=
 padding-left:10px; vertical-align:top;"><table border=3D"0" cellpadding=3D=
"0" cellspacing=3D"0" style=3D"width:100%;"><tbody><tr><td style=3D"width:6=
0px; vertical-align:top;"><img alt=3D"Headshot of Roshan Bastian" class=3D"=
speaker-headshot" style=3D"width: 60px; height: 60px; border-radius: 50%; d=
isplay: block;" height=3D"60" width=3D"60" src=3D"https://mx.technolutions.=
net/proxy/V7rENby-I3dRIKMcLT7ieOthm0veNcvNSQGN3qDqofPCc3O-II0TThs4ceHsEkaP3=
GUSCUtYR9AzV5khO2BXL4etkOaGYxGT_9G3QqTb5QfWMFpTGm_7reizVKq8w3VY/ElOMVt9oHus=
xWwW5hdv7ocgx46EBOBAWD3rN9OAEYG0" /></td><td style=3D"padding-left:8px; ver=
tical-align:top;"><p class=3D"speaker-name" style=3D"margin:0; font-weight:=
bold; font-size:13px;">
                                        &#8203;&#8203;</td></tr></tbody></t=
able></td></tr><tr><td><span style=3D"font-weight: normal; font-family:Aria=
l,Helvetica,sans-serif; font-size:14px;">Kind regards,<br /><strong>Graduat=
e Admissions</strong><br />
                                  Stevens Institute of Technology</span></t=
d></tr><!--=3D START CTA ROW=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=
=3D=3D=3D=3D=3D=3D=3D--><!--=3D END CTA ROW=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=
=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D--><!-- BLUE Event_Section_1 --></tbody=
></table><!-- // BLUE Event_Section_1 --></td></tr></tbody></table></td></t=
r><!--=3D FOOTER (inside card, matches card width) =3D=3D=3D=3D=3D=3D=3D=3D=
=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D--><tr><td align=3D"center" class=
=3D"footer-fullwidth"><table bgcolor=3D"#EEEEEE" cellpadding=3D"0" cellspac=
ing=3D"0" style=3D"border-collapse:collapse;margin:0 auto;" width=3D"600"><=
tbody><tr><td align=3D"center" style=3D"padding: 20px 25px; font-family: Ar=
ial, Helvetica, sans-serif; font-size: 12px; line-height: 1.6; color: rgb(7=
3, 72, 73); width: 100%; text-align: center;"><img alt=3D"" style=3D"width:=
 30%;" width=3D"30%" src=3D"https://mx.technolutions.net/proxy/V7rENby-I3dR=
IKMcLT7ieOthm0veNcvNSQGN3qDqofPCc3O-II0TThs4ceHsEkaP3GUSCUtYR9AzV5khO2BXL86=
ajicgWmQeYh7VpN6huNdlqLv05BDC3qjWbh5iDGq0/Qm-X1daVmn5DC-oPX8gENg" /><br /><=
br /><span style=3D"color:#666666;"></span><a href=3D"https://mx.technoluti=
ons.net/ss/c/u001.I7QH9QJlytPYeTGKoyJp4-KkSDpy5K_npoETmN8HMglgB6CkEg-zghtrC=
3NcE1S_3ZiS1GE65gw4cu6vyPyPnQ/4pa/VNKrvYH6Rm6eWVDnmUWojA/h2/h001.CYmlMpoXFc=
kvs2zlnzue-WNf4NYd-oopfpnh7BBj3rU" title=3D"https://maps.app.goo.gl/d6djvDW=
7CnuntZ8Q9" style=3D"text-decoration:none;"><span style=3D"color:#666666;">=
<strong>Castle Point Terrace, Hoboken, NJ 07030</strong></span></a><span st=
yle=3D"color:#666666;"></span><br /><a href=3D"https://mx.technolutions.net=
/ss/c/u001.mv75SfSNMmSvduEMaqEjkFviUYphpQhpreHBA52x0AU/4pa/VNKrvYH6Rm6eWVDn=
mUWojA/h3/h001.48KqDuBPy12X-j3BQRIqWnzsfCQrDmdVIgqjBNTmik4" title=3D"http:/=
/www.stevens.edu" style=3D"text-decoration:none;"><span style=3D"color:#666=
666;">stevens.edu</span></a><span style=3D"color:#666666;"> | 1-888-STEVENS=
 | </span><a href=3D"mailto:graduate@stevens.edu" title=3D"mailto:graduate@=
stevens.edu" style=3D"text-decoration:none;"><span style=3D"color:#666666;"=
>graduate@stevens.edu</span></a><span style=3D"color:#666666;"></span><br /=
><!-- Social icons row (bottom) --><table align=3D"center" border=3D"0" cel=
lpadding=3D"0" cellspacing=3D"0" class=3D"footer-center" style=3D"margin:10=
px auto 0 auto;"><tbody><tr><td align=3D"center" style=3D"padding-right: 6p=
x;"><a href=3D"https://mx.technolutions.net/ss/c/u001.mv75SfSNMmSvduEMaqEjk=
ArbkciIojreHAkuKLRzhinTThGTorkfwF-zcK9LLi-N/4pa/VNKrvYH6Rm6eWVDnmUWojA/h4/h=
001.hQ0DiZ5hKF2CfxJU6sP-lY31qapyqgzDKsji1ALChSY" title=3D"http://www.facebo=
ok.com/Stevens1870"><img alt=3D"Facebook" style=3D"width: 24px; height: 24p=
x; border: 0px; display: block;" height=3D"24" width=3D"24" src=3D"https://=
mx.technolutions.net/proxy/V7rENby-I3dRIKMcLT7ieOthm0veNcvNSQGN3qDqofPCc3O-=
II0TThs4ceHsEkaP3GUSCUtYR9AzV5khO2BXL0JAwP2L1X_2zT9B4b3GI4E8NQxX_bGQv0_Pmlr=
5GYSL/31MGACpAyRq5fqM1ZDKr3KD4w7UGay8q1Ma_55y9iJ0" /></a></td><td align=3D"=
center" style=3D"padding-right: 6px;"><a href=3D"https://mx.technolutions.n=
et/ss/c/u001.aZtlQtJOBbTRp5wW0JgumtRseBMW9iZ0Pduoh2MfMz2DRKKYdKkhqzb5uZsM30=
zo/4pa/VNKrvYH6Rm6eWVDnmUWojA/h5/h001.NuShqSVqi76GGiViPBtijq-ieN78aAYFPllUR=
rMk9vo" title=3D"http://twitter.com/followstevens"><img alt=3D"Twitter" sty=
le=3D"width: 24px; height: 24px; border: 0px; display: block;" height=3D"24=
" width=3D"24" src=3D"https://mx.technolutions.net/proxy/V7rENby-I3dRIKMcLT=
7ieOthm0veNcvNSQGN3qDqofPCc3O-II0TThs4ceHsEkaP3GUSCUtYR9AzV5khO2BXL0JAwP2L1=
X_2zT9B4b3GI4E8NQxX_bGQv0_Pmlr5GYSL/R0oPG7b24I-EDxW8qXfx16mjtzFama57TjMEzgt=
TCGc" /></a></td><td align=3D"center" style=3D"padding-right: 6px;"><a href=
=3D"https://mx.technolutions.net/ss/c/u001.jxaZRujEt1lhE0WEp4QewP0BSOFx9sM6=
ZdO5ggLiKeE1tfb9qPXwclvo-3t59JZAQwZpI9vXFtYrRKx41CfIIlIf8OWq08spu81s843SwDE=
/4pa/VNKrvYH6Rm6eWVDnmUWojA/h6/h001.2gKoQzOygH5cRoj9Sa3p5nRbFtCQTRK5atlZqby=
svYQ" title=3D"https://www.linkedin.com/company/stevens-institute-of-techno=
logy"><img alt=3D"LinkedIn" style=3D"width: 24px; height: 24px; border: 0px=
; display: block;" height=3D"24" width=3D"24" src=3D"https://mx.technolutio=
ns.net/proxy/V7rENby-I3dRIKMcLT7ieOthm0veNcvNSQGN3qDqofPCc3O-II0TThs4ceHsEk=
aP3GUSCUtYR9AzV5khO2BXL0JAwP2L1X_2zT9B4b3GI4E8NQxX_bGQv0_Pmlr5GYSL/MqbSZ0h7=
dpxvykVSrjBjy0uferjsg2CSVNLVCrmcQTA" /></a></td><td align=3D"center" style=
=3D"padding-right: 6px;"><a href=3D"https://mx.technolutions.net/ss/c/u001.=
fFHVASszD5bbJFKuGtF21GngIs9YL5zBZ9mFvkdeqbmtL6IF8D43YlfmWgx2Su0E/4pa/VNKrvY=
H6Rm6eWVDnmUWojA/h7/h001.gdtA6UPZbHSNSLH1Gpa3ZXLfLl8RaMq3FeW84ZorCrg" title=
=3D"http://instagram.com/followstevens/"><img alt=3D"Instagram" style=3D"wi=
dth: 24px; height: 24px; border: 0px; display: block;" height=3D"24" width=
=3D"24" src=3D"https://mx.technolutions.net/proxy/V7rENby-I3dRIKMcLT7ieOthm=
0veNcvNSQGN3qDqofPCc3O-II0TThs4ceHsEkaP3GUSCUtYR9AzV5khO2BXL0JAwP2L1X_2zT9B=
4b3GI4E8NQxX_bGQv0_Pmlr5GYSL/c2NgxX1ZLzljF8nsh2oa5wE-NNUstJMS0BVl2v-q0jg" /=
></a></td><td align=3D"center"><a href=3D"https://mx.technolutions.net/ss/c=
/u001.mv75SfSNMmSvduEMaqEjkKHo2Ey_1N_h4fvW58dMA6JzOOcoD6qe4uh21_YRCkenhlMZb=
kG5Wphr1gV7TOmanw/4pa/VNKrvYH6Rm6eWVDnmUWojA/h8/h001.NJMGlJQ2wStRzevPUPyB3q=
P33kG263sXl8hMXLIedGw" title=3D"http://www.youtube.com/user/EdwinAStevens70=
"><img alt=3D"YouTube" style=3D"width: 24px; height: 24px; border: 0px; dis=
play: block;" height=3D"24" width=3D"24" src=3D"https://mx.technolutions.ne=
t/proxy/V7rENby-I3dRIKMcLT7ieOthm0veNcvNSQGN3qDqofPCc3O-II0TThs4ceHsEkaP3GU=
SCUtYR9AzV5khO2BXL0JAwP2L1X_2zT9B4b3GI4E8NQxX_bGQv0_Pmlr5GYSL/UF72IF1JMvqUT=
c6iDkAEVFRk_2kgN5EsIgUnZCn7arg" /></a></td></tr></tbody></table></td></tr><=
/tbody></table></td></tr><!--=3D END FOOTER (inside card, matches card widt=
h) =3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D--></t=
body></table><!--=3D END WRAPPER TABLE=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=
=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D--></td></tr></tbody></table></td></tr></tbod=
y></table><!--=3D END FULL PAGE TABLE=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D=
=3D=3D=3D=3D=3D=3D=3D=3D=3D=3D--><div class=3D"deliver_unsubscribe" style=
=3D"opacity: 0.75; clear: both; color: #333333; font-family: Arial, Helveti=
ca, sans-serif; font-size: 11px; margin: 25px auto; width: auto; max-width:=
 600px; background-color: #fafafa; text-align: center; padding: 5px;">This =
email was sent to parvasheta24@gmail.com by Stevens Institute of Technology=
.<br /><a href=3D"https://mx.technolutions.net/ss/c/u001.Jg_sqgPJs1lCjHcCRs=
BB8fEl66DSe5MnVK3g7OCgm1YUgeMDg0hqM2GIgCAgGP8PHzZAVa1iRUWbPZI0ZWKesk0GH7vHc=
tG7yBwV8RFtix9uHoZeX6uM2xsofhdWSwS2/4pa/VNKrvYH6Rm6eWVDnmUWojA/h9/h001.kIPM=
d8MDDsNisZGZFuIq0y2pqmqlDJ9IB6ZVZcKUcXk">Unsubscribe</a> from Stevens Insti=
tute of Technology.</div>
<img src=3D"https://mx.technolutions.net/ss/o/u001.l7ggKG0fqQ6Gg4_ean_HcQ/4=
pa/VNKrvYH6Rm6eWVDnmUWojA/ho.gif" alt=3D"" width=3D"1" height=3D"1" border=
=3D"0" style=3D"height:1px !important;width:1px !important;border-width:0 !=
important;margin-top:0 !important;margin-bottom:0 !important;margin-right:0=
 !important;margin-left:0 !important;padding-top:0 !important;padding-botto=
m:0 !important;padding-right:0 !important;padding-left:0 !important;"/></bo=
dy>
</html>"""
},
]
a = classify_emails_batch(state, email_objs)

marketing_agent


In [ ]:
# Vertex API smoke test with Gemini 2.5 Flash Lite (single-email sample)
from google import genai
from google.genai import types

from app.core.config import settings

sample_email = {
    "from_name": "Amazon",
    "from_address": "shipment-tracking@amazon.com",
    "subject": "Your package has shipped",
    "snippet": "Track your package and expected delivery date",
    "body": "Your order #123-4567890 has shipped. Track it with this link...",
}

client = genai.Client(vertexai=True, api_key=settings.google_api_key)

schema = {
    "type": "object",
    "properties": {
        "coarse_group": {"type": "string"},
        "confidence": {"type": "number"}
    },
    "required": ["coarse_group", "confidence"]
}

prompt = f"""
Classify this email into one coarse group:
transactional, marketing, institutional, personal, needs_human_review.

Email:
{sample_email}
"""

resp = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=schema,
        temperature=0,
    ),
)

print(resp.text)
